<div align="center">

# Jackrong-llm-finetuning-guide 🌌
### 🛠️ Module: Dataset Preparation (API Distillation)
**An Educational, End-to-End LLM Fine-Tuning Pipeline for Beginners and Developers**

This notebook currently focuses on a fine-tuning guide for **🐳 DeepSeek-V4 API Distillation Pipeline**

🌐 **Language:**  🇬🇧 **English** 

🤗 **HuggingFace:** [Jackrong](https://huggingface.co/Jackrong)

<br>

[![DeepSeek-V4](https://img.shields.io/badge/Model-DeepSeek--V4-6495ED?style=flat-square&logo=deepseek&logoColor=white)](https://www.deepseek.com/)
[![Google Colab](https://img.shields.io/badge/Environment-Google%20Colab-F9AB00?style=flat-square&logo=googlecolab&logoColor=white)](https://colab.research.google.com/)
[![CoT Reasoning](https://img.shields.io/badge/Capability-CoT%20Reasoning-blueviolet?style=flat-square)](#)
[![API Distillation](https://img.shields.io/badge/Technique-API%20Distillation-007EC6?style=flat-square)](#)
[![Hugging Face](https://img.shields.io/badge/Model%20Hub-Hugging%20Face-FFD21E?style=flat-square&logo=huggingface&logoColor=black)](https://huggingface.co/)
[![Beginner Friendly](https://img.shields.io/badge/Level-Beginner%20Friendly-brightgreen?style=flat-square)](#)

</div>

---


# 🐳 DeepSeek-V4 API Distillation Pipeline



> [!NOTE]
> This is an automated script designed to extract Chain-of-Thought (CoT) reasoning capabilities from **DeepSeek-V4-Flash/Pro**. It batches requests through the official DeepSeek API to retrieve the reasoning process (`reasoning_content`) and the final answer, formatting them into standard training datasets.

## 🚀 Key Features

- **Asynchronous & High Concurrency**: Built with `asyncio` and `httpx` to support highly efficient, large-scale data processing.
- **Multi-API-Key Support**: Enables concurrent execution using multiple API keys, automatically distributing tasks to maximize rate limits.
- **Stateful Resume**: Automatically identifies processed data IDs, allowing seamless continuation after interruptions without duplicate API usage.
- **Fault Tolerance**: Built-in exponential backoff retry strategy gracefully handles network instability, such as 429 (Rate Limit) and 5xx (Server Error).
- **Graceful Pause**: Supports safely stopping tasks via a flag file (`pause_file`), ensuring all in-flight requests complete before exiting.
- **Standardized Output**: Generates JSONL files compliant with Hugging Face specifications, automatically wrapping reasoning within `<think>` tags.

## 📖 Usage Guide

> [!IMPORTANT]
> ⚠️ **Environment Setup**: Create a `DeepSeekAPI.env` file in the project directory, and add your API keys in the format `ds1=your_key`.

1. **Configure Paths**: Update variables like `project_dir`, `input_file`, and `output_file` in the code.
2. **Smoke Test**: It is highly recommended to run the initial testing cell to ensure API connectivity and permissions are correct.
3. **Start Distillation**: Execute the main logic cell. The script will automatically process data and display real-time progress in the terminal via `tqdm`.

In [ ]:
import json
import time
from pathlib import Path
import httpx

# TODO: Update the data file path to match your local environment
env_file = Path('/Users/Jackrong/PycharmProjects/DeepSeek-v4-distill/DeepSeekAPI.env')
api_endpoint = 'https://api.deepseek.com/chat/completions'
model_name = 'deepseek-v4-flash'

test_question = '9.11 and 9.8, which number is greater? Please answer briefly.'
request_timeout_seconds = 1800


def load_one_deepseek_key(env_file: Path, key_name: str = 'ds1') -> str:
    for raw_line in env_file.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        name, value = line.split('=', 1)
        if name.strip() == key_name:
            return value.strip().strip('"').strip("'")
    raise ValueError(f'{key_name} not found in {env_file}')


api_key = load_one_deepseek_key(env_file, 'ds1')
headers = {
    'Authorization': f'Bearer {api_key}',
    'Accept': 'application/json',
    'Content-Type': 'application/json',
}
payload = {
    'model': model_name,
    'messages': [{'role': 'user', 'content': test_question}],
    'stream': False,
    'max_tokens': 16384,
    'reasoning_effort': 'high',
    'thinking': {'type': 'enabled'},
}

started = time.perf_counter()
with httpx.Client(timeout=request_timeout_seconds) as client:
    response = client.post(api_endpoint, headers=headers, json=payload)
elapsed = time.perf_counter() - started

print(f'HTTP status: {response.status_code}')
print(f'Elapsed: {elapsed:.2f}s')

if response.status_code >= 400:
    print(response.text[:4000])
    response.raise_for_status()

data = response.json()
choice = data['choices'][0]
message = choice.get('message') or {}
reasoning_content = message.get('reasoning_content') or ''
content = message.get('content') or ''

print('Model:', data.get('model'))
print('Finish reason:', choice.get('finish_reason'))
print('Usage:', json.dumps(data.get('usage', {}), ensure_ascii=False, indent=2))
print('\n--- reasoning_content ---')
print(reasoning_content[:4000] if reasoning_content else '[empty]')
print('\n--- content ---')
print(content[:4000] if content else '[empty]')

## ⚙️  Main Distillation Code Distillation Pipeline & API Schema

This section implements the core logic for high-throughput data distillation. The pipeline is designed to be **highly scalable** and follows the **OpenAI-compatible Chat Completions API** standard, with specific extensions for DeepSeek's reasoning capabilities.

### 🔌 API Request Schema
The request payload sent to the endpoint follows this structure:

```json
{
  "model": "deepseek-v4-flash",
  "messages": [{"role": "user", "content": "...prompt..."}],
  "max_tokens": 16384,
  "reasoning_effort": "high",
  "thinking": { "type": "enabled" }
}
```

> [!TIP]
> While optimized for DeepSeek, the code can be easily adapted for other OpenAI-compatible providers (like SiliconFlow, Together AI, or local vLLM instances) by adjusting the `api_endpoint` and `model_name` variables.

### 🏗️ Pipeline Architecture
The distillation workflow follows a **Producer-Consumer** pattern to maximize throughput:

1. **Initialization**: Loads API keys from `.env` and scans the output file to skip already processed samples.
2. **Work Queue**: Populates an `asyncio.Queue` with unprocessed samples.
3. **Async Workers**: Spawns multiple concurrent workers (`per_key_concurrency`) for each API key.
4. **Resilience Layer**:
   - **Exponential Backoff**: Handles `429` (Rate Limit) and `5xx` errors.
   - **Graceful Shutdown**: Monitors for a `pause_file` to exit safely without losing data.
5. **Persistence**: Writes results to JSONL files in real-time with atomic `fsync` operations.


In [ ]:
import asyncio
import hashlib
import json
import os
import random
import time
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

import httpx

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None

try:
    import nest_asyncio
    nest_asyncio.apply()
except Exception:
    pass

# =============================================================================
# ⚠️ Config - edit these values for yourself.
# =============================================================================

# TODO: Update the data file path to match your local environment
project_dir = Path('/Users/Jackrong/PycharmProjects/DeepSeek-v4-distill')
env_file = project_dir / 'DeepSeekAPI.env'
input_file = project_dir / 'Jackrong_GLM-5.1-Reasoning-1M-Cleaned_main_seed_questions_20000.jsonl'
output_file = project_dir / 'Jackrong_DeepSeek-V4-Flash-Reasoning-20K-Distill_main.jsonl'
error_file = project_dir / 'Jackrong_DeepSeek-V4-Flash-Reasoning-20K-Distill_errors.jsonl'
pause_file = project_dir / 'DEEPSEEK_DISTILL_PAUSE.flag'

api_endpoint = 'https://api.deepseek.com/chat/completions'
model_name = 'deepseek-v4-flash'
teacher_model_name = 'DeepSeek-V4-Flash'

# DeepSeek's public docs list a 1M model context. The API does not expose a
# separate "context window" knob, so this notebook sends 16384 as max_tokens.
max_output_tokens = 16384
reasoning_effort = 'high'
thinking = {'type': 'enabled'}

key_prefix = 'ds'
total_api_keys = 10
per_key_concurrency = 10
require_exact_key_count = True

request_timeout_seconds = 1800
max_retries = 5
retryable_status_codes = {408, 409, 425, 429, 500, 502, 503, 504}
retry_base_seconds = 2.0
retry_max_seconds = 60.0

# Set limit to a small integer for a smoke test, then restore to None.
limit: Optional[int] = None
fsync_every_n_records = 1

# Temperature/top_p/presence_penalty/frequency_penalty are intentionally omitted:
# DeepSeek docs state they are not supported in thinking mode and have no effect.

# =============================================================================
# Helpers.
# =============================================================================
def parse_env_file(path: Path) -> Dict[str, str]:
    if not path.exists():
        raise FileNotFoundError(f'Env file not found: {path}')

    values: Dict[str, str] = {}
    for raw_line in path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key:
            values[key] = value
    return values


def load_api_keys(path: Path, prefix: str, total: int) -> List[Tuple[str, str]]:
    env = parse_env_file(path)
    keys: List[Tuple[str, str]] = []
    missing: List[str] = []

    for idx in range(1, total + 1):
        name = f'{prefix}{idx}'
        key = env.get(name, '').strip()
        if key:
            keys.append((name, key))
        else:
            missing.append(name)

    if require_exact_key_count and missing:
        raise ValueError(f'Missing required API keys in {path}: {missing}')
    if not keys:
        raise ValueError(f'No API keys loaded from {path}')
    return keys


def iter_jsonl(path: Path) -> Iterable[Dict[str, Any]]:
    with path.open('r', encoding='utf-8') as handle:
        for line_no, line in enumerate(handle, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f'Invalid JSON at {path}:{line_no}: {exc}') from exc
            if isinstance(obj, dict):
                yield obj


def stable_id(text: str) -> str:
    return hashlib.md5(text.encode('utf-8')).hexdigest()


def sample_id(sample: Dict[str, Any]) -> str:
    prompt = str(sample.get('problem') or sample.get('input') or '')
    return str(sample.get('id') or stable_id(prompt))


def sample_prompt(sample: Dict[str, Any]) -> str:
    return str(sample.get('problem') or sample.get('input') or '')


def sample_domain(sample: Dict[str, Any]) -> str:
    return str(sample.get('category') or sample.get('domain') or 'main')


def load_processed_ids(path: Path) -> set:
    processed = set()
    if not path.exists():
        return processed
    for row in iter_jsonl(path):
        sid = row.get('id')
        if sid:
            processed.add(str(sid))
    return processed


def build_work_items(input_file: Path, processed_ids: set) -> Tuple[List[Dict[str, Any]], int, int]:
    work: List[Dict[str, Any]] = []
    seen = set()
    duplicate_count = 0
    skipped_done_count = 0

    for sample in iter_jsonl(input_file):
        sid = sample_id(sample)
        prompt = sample_prompt(sample).strip()
        if not prompt:
            continue
        if sid in seen:
            duplicate_count += 1
            continue
        seen.add(sid)
        if sid in processed_ids:
            skipped_done_count += 1
            continue
        work.append(sample)
        if limit is not None and len(work) >= limit:
            break

    return work, duplicate_count, skipped_done_count


def format_reasoning_output(reasoning: str, answer: str) -> str:
    reasoning = (reasoning or '').strip()
    answer = (answer or '').strip()
    if reasoning:
        return f'<think>\n{reasoning}\n</think>\n\n{answer}' if answer else f'<think>\n{reasoning}\n</think>'
    return answer


def build_hf_record(sample: Dict[str, Any], api_result: Dict[str, Any]) -> Dict[str, Any]:
    prompt = sample_prompt(sample)
    reasoning = api_result.get('reasoning_content') or ''
    answer = api_result.get('content') or ''
    output = format_reasoning_output(reasoning, answer)
    usage = api_result.get('usage') or {}

    return {
        'id': sample_id(sample),
        'conversations': [
            {'from': 'human', 'value': prompt},
            {'from': 'gpt', 'value': output},
        ],
        'input': prompt,
        'output': output,
        'domain': sample_domain(sample),
        'meta': {
            'input_tokens': usage.get('prompt_tokens'),
            'output_tokens': usage.get('completion_tokens'),
            'teacher_model': teacher_model_name,
        },
    }


def make_error_record(sample: Dict[str, Any], key_name: str, error: str) -> Dict[str, Any]:
    return {
        'id': sample_id(sample),
        'problem': sample_prompt(sample),
        'domain': sample_domain(sample),
        'key_name': key_name,
        'error': error,
        'time': time.strftime('%Y-%m-%d %H:%M:%S'),
    }


def backoff_delay(attempt: int) -> float:
    base = min(retry_max_seconds, retry_base_seconds * (2 ** max(0, attempt - 1)))
    return base * random.uniform(0.75, 1.25)


async def call_deepseek(client: httpx.AsyncClient, api_key: str, prompt: str) -> Dict[str, Any]:
    headers = {'Authorization': f'Bearer {api_key}'}
    payload = {
        'model': model_name,
        'messages': [{'role': 'user', 'content': prompt}],
        'stream': False,
        'max_tokens': max_output_tokens,
        'reasoning_effort': reasoning_effort,
        'thinking': thinking,
    }

    last_error = 'unknown error'
    for attempt in range(1, max_retries + 1):
        try:
            response = await client.post(api_endpoint, headers=headers, json=payload)
            if response.status_code >= 400:
                body = response.text[:2000]
                last_error = f'HTTP {response.status_code}: {body}'
                if response.status_code in retryable_status_codes and attempt < max_retries:
                    await asyncio.sleep(backoff_delay(attempt))
                    continue
                raise RuntimeError(last_error)

            data = response.json()
            choice = data['choices'][0]
            message = choice.get('message') or {}
            content = message.get('content') or ''
            reasoning_content = message.get('reasoning_content') or ''
            if not content.strip() and not reasoning_content.strip():
                last_error = 'empty content and empty reasoning_content'
                if attempt < max_retries:
                    await asyncio.sleep(backoff_delay(attempt))
                    continue
                raise RuntimeError(last_error)

            return {
                'content': content,
                'reasoning_content': reasoning_content,
                'usage': data.get('usage') or {},
                'model': data.get('model'),
                'finish_reason': choice.get('finish_reason'),
            }
        except (httpx.TimeoutException, httpx.TransportError) as exc:
            last_error = f'{type(exc).__name__}: {exc}'
            if attempt < max_retries:
                await asyncio.sleep(backoff_delay(attempt))
                continue
            raise RuntimeError(last_error) from exc
        except (KeyError, IndexError, ValueError) as exc:
            last_error = f'bad response schema: {exc}'
            if attempt < max_retries:
                await asyncio.sleep(backoff_delay(attempt))
                continue
            raise RuntimeError(last_error) from exc

    raise RuntimeError(last_error)


async def write_jsonl(handle, record: Dict[str, Any], lock: asyncio.Lock, counter: Dict[str, int], key: str) -> None:
    async with lock:
        handle.write(json.dumps(record, ensure_ascii=False) + '\n')
        handle.flush()
        counter[key] = counter.get(key, 0) + 1
        if fsync_every_n_records and counter[key] % fsync_every_n_records == 0:
            os.fsync(handle.fileno())


async def worker(
    key_name: str,
    api_key: str,
    slot: int,
    queue: asyncio.Queue,
    client: httpx.AsyncClient,
    out_handle,
    err_handle,
    write_lock: asyncio.Lock,
    pbar,
    stats: Dict[str, int],
) -> None:
    worker_name = f'{key_name}:{slot:02d}'
    while True:
        if pause_file.exists():
            break
        try:
            sample = queue.get_nowait()
        except asyncio.QueueEmpty:
            break

        try:
            result = await call_deepseek(client, api_key, sample_prompt(sample))
            record = build_hf_record(sample, result)
            await write_jsonl(out_handle, record, write_lock, stats, 'success')
        except Exception as exc:
            error_record = make_error_record(sample, key_name, str(exc))
            await write_jsonl(err_handle, error_record, write_lock, stats, 'error')
        finally:
            queue.task_done()
            if pbar is not None:
                pbar.update(1)

    if pause_file.exists():
        print(f'{worker_name} paused after finishing its in-flight request.')


async def run_distillation() -> None:
    if not input_file.exists():
        raise FileNotFoundError(f'Input file not found: {input_file}')

    api_keys = load_api_keys(env_file, key_prefix, total_api_keys)
    processed_ids = load_processed_ids(output_file)
    work_items, duplicate_count, skipped_done_count = build_work_items(input_file, processed_ids)

    print(f'Loaded API keys: {[name for name, _ in api_keys]}')
    print(f'Output file: {output_file}')
    print(f'Error file:  {error_file}')
    print(f'Pause file:  {pause_file}')
    print(f'Already done: {len(processed_ids)} | skipped done: {skipped_done_count} | input duplicates skipped: {duplicate_count}')
    print(f'Queued: {len(work_items)} | concurrency: {len(api_keys)} keys x {per_key_concurrency} = {len(api_keys) * per_key_concurrency}')

    if pause_file.exists():
        print(f'Pause file exists. Remove it before starting: {pause_file}')
        return
    if not work_items:
        print('No new questions to process.')
        return

    queue: asyncio.Queue = asyncio.Queue()
    for item in work_items:
        queue.put_nowait(item)

    timeout = httpx.Timeout(request_timeout_seconds, connect=60.0, read=request_timeout_seconds, write=60.0, pool=60.0)
    limits = httpx.Limits(
        max_connections=len(api_keys) * per_key_concurrency + 20,
        max_keepalive_connections=len(api_keys) * per_key_concurrency + 20,
    )
    headers = {'Accept': 'application/json', 'Content-Type': 'application/json'}
    write_lock = asyncio.Lock()
    stats: Dict[str, int] = {'success': 0, 'error': 0}
    pbar = tqdm(total=len(work_items), desc='DeepSeek distillation') if tqdm is not None else None

    started_at = time.time()
    try:
        async with httpx.AsyncClient(timeout=timeout, limits=limits, headers=headers) as client:
            with output_file.open('a', encoding='utf-8') as out_handle, error_file.open('a', encoding='utf-8') as err_handle:
                tasks = [
                    asyncio.create_task(worker(key_name, api_key, slot, queue, client, out_handle, err_handle, write_lock, pbar, stats))
                    for key_name, api_key in api_keys
                    for slot in range(1, per_key_concurrency + 1)
                ]
                await asyncio.gather(*tasks)
                out_handle.flush(); os.fsync(out_handle.fileno())
                err_handle.flush(); os.fsync(err_handle.fileno())
    finally:
        if pbar is not None:
            pbar.close()

    elapsed = time.time() - started_at
    remaining = queue.qsize()
    if pause_file.exists():
        print(f'Paused cleanly. Success: {stats.get("success", 0)}, errors: {stats.get("error", 0)}, remaining in queue: {remaining}, elapsed: {elapsed:.1f}s')
        print(f'Remove {pause_file} and rerun this cell to continue.')
    else:
        print(f'Finished run. Success: {stats.get("success", 0)}, errors: {stats.get("error", 0)}, remaining in queue: {remaining}, elapsed: {elapsed:.1f}s')


# To pause cleanly while requests are running, create this file from another terminal:
# The notebook will stop assigning new work and will exit after in-flight calls are written.
# Remove the file and rerun this cell to resume from output_file.
await run_distillation()